# 02 — Your First LLM Call from Python

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Use the `ask_llm()` wrapper to call any configured provider.
2. Send a CA/finance question and read the structured answer.
3. Ask the model for **JSON** so the result can be used in pandas.
4. Build a **prompt template** for repeated tasks.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
from src.llm_client import ask_llm, ask_llm_json

## 2.1 — A plain finance question

In [ ]:
print(ask_llm('What is deferred tax, in one paragraph?'))

## 2.2 — Ask three CA-style questions in a loop

In [ ]:
questions = [
    'Give 3 audit risks specific to a manufacturing company in Nepal.',
    'List the typical documentation expected to support a related-party transaction.',
    'Explain (briefly) the difference between TDS and Advance Tax in Nepal.'
]
for q in questions:
    print('Q:', q)
    print('A:', ask_llm(q))
    print('-' * 60)

## 2.3 — Structured JSON output for downstream automation

In [ ]:
import pandas as pd
memo = '''\
1. Bank reconciliation has stale cheques older than 6 months.
2. Two vendors lack PAN/VAT registration in the master.
3. Inventory item FG-509 is 700+ days old and not provided for.
4. Related-party purchases without benchmark pricing.
'''
obj = ask_llm_json(
    'Extract a JSON array of findings from the memo. Each item has "finding", '
    '"category" (one of Finance, Procurement, Inventory, Compliance), and '
    '"recommended_action".\n\nMemo:\n' + memo
)
print(obj)
if isinstance(obj, list):
    df = pd.DataFrame(obj)
    display(df)

## 2.4 — Build a prompt template (DRY)

We wrap a repeated prompt in a function for cleanliness.

In [ ]:
def summarise_to_bullets(text, n=5):
    prompt = f'Summarise the text below in exactly {n} short bullet points for a CA audience.\n\n{text}'
    return ask_llm(prompt, system='You are a Nepali CA assistant. Be concise.')

memo = ('During our planning meeting we identified five risks. Revenue cut-off is the most '
        'significant given the Q4 spike. Inventory existence is high risk due to the new '
        'production line. Related-party transactions need careful inspection. Loan DSCR has '
        'tight headroom. Going concern needs revisiting.')
print(summarise_to_bullets(memo, n=5))

## Expected output

* Section 2.3 should give a JSON array → DataFrame with the four findings.
* Section 2.4 should give exactly five bullets.


## Exercise

Pick one of your routine emails (vendor follow-up, audit-confirmation letter) and turn it into a reusable `ask_llm` template. Run it on three different inputs.


## Common errors

| Symptom | Fix |
|---|---|
| `obj['_parse_error']` shows up | The model added commentary. Re-run; if persistent, tighten the prompt: *only JSON, no markdown fences*. |
| 429 / rate limit | Wait a few seconds. For class delivery, lower `temperature` and `k`. |


## ⚠️ Professional caution

Do not paste real client emails, balance sheets, or PII into a public LLM provider without engagement-letter authority. Synthetic data only at this stage.